# A GPT-2 Scale End-to-End Distributed LLM Pipeline

**Objective:** This is an end-to-end, large-scale Language Model (LLM) training pipeline from scratch, for a scalable GPT-2 style Transformer model using TensorFlow. It highlights critical system-level engineering decisions required for modern LLM workloads including addressing compute bottlenecks, memory constraints, and pipeline throughput.

**Core System Optimizations Demonstrated:**
*   **Distributed Compute:** Multi-GPU orchestration via `tf.distribute.MirroredStrategy` for synchronous data parallelism.
*   **Memory Virtualization:** Implementation of Gradient Accumulation to simulate massive global batch sizes (512) while strictly adhering to physical GPU VRAM limits.
*   **Throughput Maximization:** `tf.data` asynchronous streaming (to prevent GPU starvation) and `bfloat16` mixed-precision training (to double FLOPs and halve memory overhead).

## 1. Hardware Provisioning & Distributed Strategy
Initialization of the compute environment. We utilize `MirroredStrategy` to synchronize gradients across all available GPUs, ensuring optimal parallelization and scaling the effective batch size mathematically.

In [ ]:
!pip install -q --upgrade keras-hub tensorflow-datasets

In [ ]:
import tensorflow as tf

# 1. Initialize multi-GPU tensorflow object to handle distributing training across GPUs
strategy = tf.distribute.MirroredStrategy()
print(f'Number of GPUs in sync: {strategy.num_replicas_in_sync}')

# Scale the batch size by the number of GPUs
GLOBAL_BATCH_SIZE = 8 * strategy.num_replicas_in_sync

# Gradient Accumulation Parameters
ACCUMULATION_STEPS = 8 # Accumulate 8 steps to simulate a 64 batch size per GPU
EFFECTIVE_BATCH_SIZE = GLOBAL_BATCH_SIZE * ACCUMULATION_STEPS

print(f'Global Batch Size (Per Step): {GLOBAL_BATCH_SIZE}')
print(f'Effective Batch Size with Accumulation: {EFFECTIVE_BATCH_SIZE}')


Number of GPUs in sync: 8
Global Batch Size (Per Step): 64
Effective Batch Size with Accumulation: 512


In [ ]:
import tensorflow as tf

# Check available physical devices
gpus = tf.config.list_physical_devices('GPU')
print(f"Physical GPUs detected: {len(gpus)}")
for gpu in gpus:
    print(f" - {gpu}")

# Confirm strategy details
print(f"\nStrategy num_replicas_in_sync: {strategy.num_replicas_in_sync}")

Physical GPUs detected: 8
 - PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
 - PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')
 - PhysicalDevice(name='/physical_device:GPU:2', device_type='GPU')
 - PhysicalDevice(name='/physical_device:GPU:3', device_type='GPU')
 - PhysicalDevice(name='/physical_device:GPU:4', device_type='GPU')
 - PhysicalDevice(name='/physical_device:GPU:5', device_type='GPU')
 - PhysicalDevice(name='/physical_device:GPU:6', device_type='GPU')
 - PhysicalDevice(name='/physical_device:GPU:7', device_type='GPU')

Strategy num_replicas_in_sync: 8


## 2. Data Engineering & Pipeline Optimization
Implementing a robust, high-throughput data pipeline. Standard in-memory datasets fail at LLM scale.

**Architectural Choices:**
* **Streaming & Prefetching:** Using `tf.data.AUTOTUNE` to asynchronously fetch data from internal TFDS mirrors, ensuring the GPUs never idle waiting for disk I/O.
* **Native Tokenization:** Leveraging Keras Hub for highly optimized, graph-compatible BPE tokenization.

In [ ]:
# Load dataset

import tensorflow as tf
import tensorflow_datasets as tfds
import keras_hub

# Load Wiki40b (English) natively from TFDS internal mirrors
ds_data = tfds.load('wiki40b/en', split='train')

GPT2_T = 256 # Sequence length

# Instantiate official GPT-2 BPE Tokenizer from Keras Hub
print("Loading Keras Hub GPT-2 Tokenizer...")
tokenizer = keras_hub.models.Tokenizer.from_preset("gpt2_base_en")
gpt2_vocab_size = tokenizer.vocabulary_size()
print(f"GPT-2 Vocab Size: {gpt2_vocab_size}")

def prepare_dataset(example):
    # Tokenize the text natively
    tokens = tokenizer(example['text'])
    tokens = tf.cast(tokens, tf.int32)

    # Truncate or pad to exactly GPT2_T + 1 length
    seq_length = GPT2_T + 1
    tokens = tokens[:seq_length]
    pad_length = tf.maximum(0, seq_length - tf.shape(tokens)[0])

    # Pad with 0s if the sequence is shorter than GPT2_T + 1
    tokens = tf.pad(tokens, [[0, pad_length]], constant_values=0)

    inputs = tokens[:-1]
    targets = tokens[1:]

    # Ensure static shapes for Auto-XLA optimization
    inputs.set_shape([GPT2_T])
    targets.set_shape([GPT2_T])

    return inputs, targets

# Create token streaming pipeline
train_ds_streaming = (
    ds_data
    .map(prepare_dataset, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(GLOBAL_BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE)
)

print("Streaming dataset ready! (Replaces in-memory arrays)")


Loading Keras Hub GPT-2 Tokenizer...
GPT-2 Vocab Size: 50257
Streaming dataset ready! (Replaces in-memory arrays)


In [ ]:
import tensorflow_datasets as tfds

# Get builder information for wiki40b/en without fully downloading it
builder = tfds.builder('wiki40b/en')

# Display dataset metadata
print(f"Total Dataset Size: {builder.info.dataset_size}")
if 'train' in builder.info.splits:
    print(f"Number of training examples: {builder.info.splits['train'].num_examples}")


Total Dataset Size: 9.91 GiB
Number of training examples: 2926536


## 3. Model Architecture & System Optimizations
Defining the Transformer architecture and optimizations.

**Key Optimizations:**
* **Mixed Precision:** Enforcing `bfloat16` to prevent numerical underflow/overflow (NaNs) while maximizing modern Tensor Core utilization.
* **Weight Tying:** Structurally sharing embedding weights with the final output projection layer to significantly reduce total parameter count and memory footprint.

In [ ]:
# Mixed precision

from tensorflow.keras import mixed_precision

# Upgraded to mixed_bfloat16 to prevent 'inf' loss (numerical overflow)
policy = mixed_precision.Policy('mixed_bfloat16')
mixed_precision.set_global_policy(policy)

print('Compute dtype: %s' % policy.compute_dtype)
print('Variable dtype: %s' % policy.variable_dtype)

Compute dtype: bfloat16
Variable dtype: float32


In [ ]:
# Model build
import tensorflow as tf

# GPT-2 Small Hyperparameters
gpt2_d_model = 768
gpt2_num_layers = 12
gpt2_num_heads = 12
gpt2_ffn_dim = 3072 # Typically 4x d_model

# Everything must be inside the scope to be mirrored across GPUs
with strategy.scope():
    gpt2_input = tf.keras.Input(shape=(GPT2_T,))

    # 1. Embeddings (Separated to access weights for Weight Tying)
    word_embedding_layer = tf.keras.layers.Embedding(gpt2_vocab_size, gpt2_d_model, mask_zero=True)
    position_embedding_layer = tf.keras.layers.Embedding(GPT2_T, gpt2_d_model)

    # Broadcast position embeddings across batch
    length = tf.shape(gpt2_input)[-1]
    positions = tf.range(start=0, limit=length, delta=1)
    x = word_embedding_layer(gpt2_input) + position_embedding_layer(positions)
    x = tf.keras.layers.Dropout(0.1)(x)

    # NanoGPT style Pre-Norm & GELU blocks
    def nanogpt_attention_block(x):
        nx = tf.keras.layers.LayerNormalization(epsilon=1e-5)(x) # Pre-Norm
        attn = tf.keras.layers.MultiHeadAttention(
            num_heads=gpt2_num_heads, key_dim=gpt2_d_model // gpt2_num_heads
        )(nx, nx, use_causal_mask=True)
        return tf.keras.layers.Add()([x, tf.keras.layers.Dropout(0.1)(attn)])

    def nanogpt_ffn_block(x):
        nx = tf.keras.layers.LayerNormalization(epsilon=1e-5)(x) # Pre-Norm
        ffn = tf.keras.layers.Dense(gpt2_ffn_dim, activation='gelu')(nx) # GELU Activation
        ffn = tf.keras.layers.Dense(gpt2_d_model)(ffn)
        return tf.keras.layers.Add()([x, tf.keras.layers.Dropout(0.1)(ffn)])

    # 2. 12 Transformer Blocks
    for _ in range(gpt2_num_layers):
        x = nanogpt_attention_block(x)
        x = nanogpt_ffn_block(x)

    x = tf.keras.layers.LayerNormalization(epsilon=1e-5)(x) # Final norm before output

    # 3. Output Projection (Weight Tying)
    class TiedOutputProjection(tf.keras.layers.Layer):
        def __init__(self, embedding_layer, **kwargs):
            super().__init__(**kwargs)
            self.embedding_layer = embedding_layer

        def call(self, inputs):
            # Multiply by the transposed embedding weights to get logits
            # Cast the float32 weights to the compute dtype (bfloat16) to avoid type mismatch
            return tf.matmul(inputs, tf.cast(self.embedding_layer.embeddings, inputs.dtype), transpose_b=True)

    logits = TiedOutputProjection(word_embedding_layer)(x)
    gpt2_outputs = tf.keras.layers.Activation('softmax')(logits)

    # 4. Optimizer and Compilation
    gpt2_lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=2.5e-4,
        decay_steps=5000,
        alpha=0.1
    )

    gpt2_optimizer = tf.keras.optimizers.AdamW(
        learning_rate=gpt2_lr_schedule,
        weight_decay=0.01
    )

    gpt2_model = tf.keras.Model(inputs=gpt2_input, outputs=gpt2_outputs)
    gpt2_model.compile(optimizer=gpt2_optimizer, loss='sparse_categorical_crossentropy', jit_compile=False)


ResourceExhaustedError: {{function_node __wrapped__StatelessRandomUniformV2_device_/job:localhost/replica:0/task:0/device:GPU:0}} OOM when allocating tensor with shape[50257,768] and type float on /job:localhost/replica:0/task:0/device:GPU:0 by allocator GPU_0_bfc [Op:StatelessRandomUniformV2] name: 

## 4. Distributed Training Orchestration
Implemented a custom training loop to enable **Gradient Accumulation** across multiple GPUs. The built-in model method is not compatible with memory optimization functions.

Gradient Accumulation allows you to break up batches into sub-batches for to mimic a larger global batch size that wouldn't all fit in a single GPU.

In [ ]:
# Distributed Training Setup
import tensorflow as tf

with strategy.scope():
    # 1. Define loss for distributed training
    loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=False, reduction=tf.keras.losses.Reduction.NONE)

    def compute_loss(labels, predictions):
        per_example_loss = loss_object(labels, predictions)
        return tf.nn.compute_average_loss(per_example_loss, global_batch_size=GLOBAL_BATCH_SIZE * GPT2_T)

    # 2. Create Gradient Accumulators
    gradient_accumulators = [tf.Variable(tf.zeros_like(v), trainable=False) for v in gpt2_model.trainable_variables]

    def reset_accumulators():
        for acc in gradient_accumulators:
            acc.assign(tf.zeros_like(acc))

    # 3. Math Step: Compute and Accumulate Gradients locally
    @tf.function
    def forward_backward_pass(inputs, targets):
        with tf.GradientTape() as tape:
            predictions = gpt2_model(inputs, training=True)
            loss = compute_loss(targets, predictions)
            # Scale loss down by accumulation steps so the gradients sum up correctly
            scaled_loss = loss / tf.cast(ACCUMULATION_STEPS, loss.dtype)

        gradients = tape.gradient(scaled_loss, gpt2_model.trainable_variables)

        # Accumulate gradients inside the accumulator variables
        for acc, grad in zip(gradient_accumulators, gradients):
            if grad is not None:
                acc.assign_add(grad)

        return loss

    # 4. Apply Step
    def apply_accumulated_gradients():
        gpt2_optimizer.apply_gradients(zip(gradient_accumulators, gpt2_model.trainable_variables))

    @tf.function
    def distributed_accumulate_step(dist_inputs, dist_targets):
        per_replica_losses = strategy.run(forward_backward_pass, args=(dist_inputs, dist_targets))
        return strategy.reduce(tf.distribute.ReduceOp.SUM, per_replica_losses, axis=None)

    @tf.function
    def distributed_apply_and_reset():
        strategy.run(apply_accumulated_gradients)
        strategy.run(reset_accumulators)

print("Distributed training functions successfully restored!")

Distributed training functions successfully restored!


### 4.1 Training Execution & Auto-Saving Checkpoints
Initiating the primary training loop. This phase handles the asynchronous dataset iteration across the distributed strategy and manages the application of accumulated gradients. Here I added an automated checkpointing logic to ensure trained weights were saved on an ongoing basis, in the case of system or hardware failures.

In [ ]:
# Training loop

import time
import tensorflow as tf

# Recover from the automatic checkpoints instead of the overwritten .h5 file!
# Use a unique RUN_ID to avoid conflicts if running multiple instances in the same runtime
RUN_ID = "instance_1"
checkpoint_dir = f"./gpt2_checkpoints_{RUN_ID}"

global_step = tf.Variable(0, dtype=tf.int64, trainable=False)
current_epoch_var = tf.Variable(0, dtype=tf.int64, trainable=False)

checkpoint = tf.train.Checkpoint(
    optimizer=gpt2_optimizer,
    model=gpt2_model,
    global_step=global_step,
    current_epoch=current_epoch_var
)
manager = tf.train.CheckpointManager(checkpoint, directory=checkpoint_dir, max_to_keep=3)

if manager.latest_checkpoint:
    checkpoint.restore(manager.latest_checkpoint).expect_partial()
    print(f"Phew! Successfully recovered weights from checkpoint: {manager.latest_checkpoint}")
    current_epoch_val = current_epoch_var.numpy()
else:
    print("Warning: No checkpoint found. Starting with random weights.")
    current_epoch_val = 0

total_epochs = 10
tokens_per_batch = GLOBAL_BATCH_SIZE * GPT2_T

print(f"Starting fresh training from Epoch {current_epoch_val + 1}...")

with strategy.scope():
    for epoch in range(current_epoch_val, total_epochs):
        print(f"\nEpoch {epoch+1}/{total_epochs}")
        start_time = time.time()

        num_batches = 0

        reset_accumulators()

        dist_epoch_dataset = strategy.experimental_distribute_dataset(train_ds_streaming)

        step_start_time = time.time()
        for dist_inputs, dist_targets in dist_epoch_dataset:

            loss = distributed_accumulate_step(dist_inputs, dist_targets)
            num_batches += 1

            if num_batches % ACCUMULATION_STEPS == 0:
                distributed_apply_and_reset()

            # Only print and force CPU-GPU sync every 50 batches
            if num_batches % 50 == 0:
                current_loss = float(loss) # Forces sync here, but only every 50 steps
                step_time = (time.time() - step_start_time) / 50.0
                tokens_per_sec = tokens_per_batch / step_time if step_time > 0 else 0
                print(f"  Batch {num_batches} Avg Time: {step_time:.3f}s - Loss: {current_loss:.4f} - Tokens/sec: {tokens_per_sec:.0f}")
                step_start_time = time.time() # Reset timer for the next 50 batches

            # --- NEW: Auto Checkpoint Saver ---
            if num_batches % 1000 == 0:
                global_step.assign(num_batches) # Only update the host variable when saving
                save_path = manager.save()
                print(f"\n>>> Saved auto-checkpoint for batch {num_batches} at {save_path}\n")

        if num_batches % ACCUMULATION_STEPS != 0:
            distributed_apply_and_reset()

        epoch_time = time.time() - start_time
        epoch_tokens_per_sec = (num_batches * tokens_per_batch) / epoch_time if epoch_time > 0 else 0

        # Sync at the end of the epoch to get the final loss
        final_loss = float(loss)
        print(f"Epoch {epoch+1} finished in {epoch_time:.2f}s - Final Loss: {final_loss:.4f} - Tokens/sec: {epoch_tokens_per_sec:.0f}")

        # --- NEW: Save at the end of each epoch ---
        current_epoch_var.assign_add(1)
        global_step.assign(0)
        save_path = manager.save()
        print(f"\n>>> Saved End-of-Epoch checkpoint at {save_path}\n")


Phew! Successfully recovered weights from checkpoint: /cns/na-d/home/risapark/gpt2_checkpoints/ckpt-38
Starting fresh training from Epoch 1...

Epoch 1/10
  Batch 50 Avg Time: 0.344s - Loss: 6.8750 - Tokens/sec: 47566
  Batch 100 Avg Time: 0.293s - Loss: 7.0000 - Tokens/sec: 55895
  Batch 150 Avg Time: 0.293s - Loss: 6.8438 - Tokens/sec: 56008
  Batch 200 Avg Time: 0.326s - Loss: 7.0938 - Tokens/sec: 50275
  Batch 250 Avg Time: 0.328s - Loss: 6.8750 - Tokens/sec: 49934
  Batch 300 Avg Time: 0.329s - Loss: 7.0625 - Tokens/sec: 49831
  Batch 350 Avg Time: 0.320s - Loss: 7.1875 - Tokens/sec: 51171
  Batch 400 Avg Time: 0.294s - Loss: 7.2812 - Tokens/sec: 55750
  Batch 450 Avg Time: 0.293s - Loss: 7.0625 - Tokens/sec: 55918
  Batch 500 Avg Time: 0.293s - Loss: 7.3125 - Tokens/sec: 55969
  Batch 550 Avg Time: 0.298s - Loss: 7.1562 - Tokens/sec: 54981
  Batch 600 Avg Time: 0.330s - Loss: 7.0938 - Tokens/sec: 49654
  Batch 650 Avg Time: 0.329s - Loss: 7.0000 - Tokens/sec: 49785


In [ ]:
# Manual model weights save
import tensorflow as tf
import os
import time

# FIX: Append a unique timestamp to the filename so we NEVER overwrite previous backups
# Changed to a local path to avoid hardcoded CNS directories
timestamp = int(time.time())
emergency_save_path = f"./gpt2_emergency_backup_{timestamp}.weights.h5"

# 1. Get the parent directory path
save_dir = os.path.dirname(emergency_save_path)

# 2. Force create the directory (and any missing parent directories)
if save_dir and not tf.io.gfile.exists(save_dir):
    print(f"Creating missing directory: {save_dir}")
    tf.io.gfile.makedirs(save_dir)

# 3. Save the weights
print("Saving model weights...")
gpt2_model.save_weights(emergency_save_path)
print(f"Success! Weights safely backed up to {emergency_save_path}")


Saving model weights...
Success! Weights safely backed up to /cns/na-d/home/risapark/gpt2_emergency_backup.weights.h5


## 5. Inference
This tests the model's generative capability given a seed text. In a production pipeline, this would serve as the foundational 'sanity check' and qualitative evaluation phase before proceeding to rigorous quantitative benchmarking (e.g., perplexity, automated downstream task evaluation).

In [ ]:
#Inference

import numpy as np
import tensorflow as tf

def generate_gpt2_text(seed_text, gen_length=50, temperature=0.7):
    print(f"--- Generating from GPT-2 Scale Model (T={GPT2_T}) ---")

    # 1. Vectorize seed text using keras_hub tokenizer
    # The tokenizer returns a list natively when given a string
    input_tokens = tokenizer(seed_text)
    if isinstance(input_tokens, tf.Tensor):
        input_tokens = input_tokens.numpy().tolist()

    for _ in range(gen_length):
        # 2. Pad or truncate to the context window (GPT2_T)
        input_padded = tf.keras.preprocessing.sequence.pad_sequences(
            [input_tokens], maxlen=GPT2_T, padding='pre', truncating='pre'
        )

        # 3. Predict the next token using the trained gpt2_model
        preds = gpt2_model.predict(input_padded, verbose=0)[0, -1, :]

        # 4. Apply temperature sampling for variety
        preds = preds / (np.sum(preds) + 1e-8) # Normalize scores
        logits = np.log(preds + 1e-8) / temperature
        next_token_id = tf.random.categorical([logits], num_samples=1)[0, 0].numpy()

        # Append chosen token ID
        input_tokens.append(next_token_id)

    # 5. Decode all tokens back to text cleanly
    # Convert back to tensor for detokenization if needed by the keras_hub tokenizer
    result_text = tokenizer.detokenize(input_tokens)
    if isinstance(result_text, tf.Tensor):
        result_text = result_text.numpy().decode('utf-8')

    return result_text

# --- TEST THE GPT-2 INFERENCE ---
prompt = "The secret to a happy life is"
print(generate_gpt2_text(prompt, gen_length=40, temperature=0.6))

--- Generating from GPT-2 Scale Model (T=256) ---
The secret to a happy life isICLE) and heNEW! atill- he! a The the a as at of! forNEW_ of asAG-'s is and hisECTIONS a was acoNEW to._

